## What to do if the quality of the model is dissatisfying?

- Should we make the model more complicated or more simple?

- Should we add more features?

- Do we simply need more data for training?

The answers to these questions are not obvious. In particular, sometimes a more complex model can lead to a deterioration in performance. Other times, adding new observations will not bring noticeable changes. In fact, the ability to make the right decision and choose the right method to improve the model distinguishes a good professional from a bad one.

----

# What is learning_curve and validation_curve from sklearn.model_selection import learning_curve, validation_curve

`learning_curve` and `validation_curve` are utilities in **scikit-learn** that help you diagnose how a machine learning model is performing.

---

## 1. `learning_curve`

**Purpose:** Shows how model performance changes as you increase the amount of training data.

### Questions it answers

* Do I need more training data?
* Is my model overfitting?
* Is my model underfitting?

### What it does

It trains the model on progressively larger subsets of the training data and computes scores on both:

* Training set
* Validation/Cross-validation set

### Example

```python
from sklearn.model_selection import learning_curve
from sklearn.ensemble import RandomForestClassifier

train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(),
    X,
    y,
    cv=5,
    train_sizes=[0.1, 0.3, 0.5, 0.7, 1.0]
)
```

### Typical interpretation

#### Overfitting

```
Training Score   = 99%
Validation Score = 75%
```

Large gap between curves → model memorizes training data.

#### Underfitting

```
Training Score   = 60%
Validation Score = 58%
```

Both scores low → model too simple.

#### Good fit

```
Training Score   = 90%
Validation Score = 88%
```

Curves converge at a high score.

---

## 2. `validation_curve`

**Purpose:** Shows how model performance changes as a specific hyperparameter changes.

### Questions it answers

* What value of `max_depth` should I use?
* What is the best `C` for SVM?
* How many neighbors should KNN have?

### What it does

Keeps the dataset fixed and repeatedly trains the model using different values of a chosen hyperparameter.

### Example

```python
from sklearn.model_selection import validation_curve
from sklearn.tree import DecisionTreeClassifier

param_range = [1, 2, 3, 5, 10, 20]

train_scores, val_scores = validation_curve(
    DecisionTreeClassifier(),
    X,
    y,
    param_name="max_depth",
    param_range=param_range,
    cv=5
)
```

---

### Typical interpretation

Suppose:

| max_depth | Train Score | Validation Score |
| --------- | ----------- | ---------------- |
| 1         | 0.70        | 0.68             |
| 3         | 0.85        | 0.83             |
| 5         | 0.95        | 0.90             |
| 10        | 1.00        | 0.84             |
| 20        | 1.00        | 0.78             |

* Small depth → underfitting
* Depth around 5 → best generalization
* Large depth → overfitting

---

## Key Difference

| Function           | Varies               | Purpose                                                      |
| ------------------ | -------------------- | ------------------------------------------------------------ |
| `learning_curve`   | Training set size    | Determine whether more data helps and diagnose bias/variance |
| `validation_curve` | Hyperparameter value | Find a good hyperparameter and diagnose over/underfitting    |

Think of it this way:

* **Learning curve:** *"What happens if I give the model more data?"*
* **Validation curve:** *"What happens if I change this parameter?"*

They are often used together during model development and tuning.


In [1]:
import warnings
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
#sharper plots
%config InlineBackend.figure_format = "retina"

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV, SGDClassifier
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore")

In [2]:
DATA_PATH = "https://raw.githubusercontent.com/Yorko/mlcourse.ai/main/data/"

In [4]:
data = pd.read_csv(DATA_PATH + "telecom_churn.csv").drop("State", axis=1)
data["International plan"] = data["International plan"].map({"Yes": 1, "No": 0})
data["Voice mail plan"] = data["Voice mail plan"].map({"Yes": 1, "No": 0})

y = data["Churn"].astype("int").values
X = data.drop("Churn", axis=1).values

## there will be seperate topic later about SGDClassifier, this is a small note

`SGDClassifier` in scikit-learn is a classifier that trains models using **Stochastic Gradient Descent (SGD)** instead of the optimization methods used by models like `LogisticRegression`.

Think of it as:

> "A scalable way to train linear classifiers on large datasets."

---

## What models can SGDClassifier train?

By changing the `loss` parameter, it can behave like different algorithms:

```python
from sklearn.linear_model import SGDClassifier
```

### Logistic Regression

```python
SGDClassifier(loss='log_loss')
```

Optimizes logistic loss, so it behaves like logistic regression.

---

### Linear SVM

```python
SGDClassifier(loss='hinge')
```

Optimizes hinge loss, giving a linear SVM.

---

### Perceptron

```python
SGDClassifier(loss='perceptron')
```

Implements the perceptron algorithm.

---

## Why use SGD?

### LogisticRegression

Typically uses solvers such as:

* `lbfgs`
* `liblinear`
* `saga`

These often need multiple passes over the entire dataset and can become slow or memory-intensive for huge datasets.

---

### SGD

Instead of computing gradients using all samples at once:

[
\nabla J(\theta)
]

it updates parameters after seeing one sample (or a small batch):

[
\theta = \theta - \eta \nabla J_i(\theta)
]

where (i) is a single training example.

This makes training much faster on large datasets.

---

## Example

Suppose you have:

```text
IMDb reviews
50,000 reviews
100,000 word features
```

Using SGD:

```python
clf = SGDClassifier(loss='log_loss')

clf.fit(X_train, y_train)
```

This is often faster and uses less memory than some traditional solvers.

---

## Why is it popular for text classification?

Text data usually has:

* Huge number of features
* Sparse matrices
* Large datasets

Examples:

* Spam detection
* Sentiment analysis
* News categorization

`SGDClassifier` works extremely well here.

That's why you'll often see:

```python
CountVectorizer
    ↓
TfidfTransformer
    ↓
SGDClassifier
```

in NLP pipelines.

---

## Difference from LogisticRegression

If you write:

```python
LogisticRegression()
```

and

```python
SGDClassifier(loss='log_loss')
```

both are trying to learn logistic regression.

The difference is **how they optimize**.

| Feature             | LogisticRegression                              | SGDClassifier       |
| ------------------- | ----------------------------------------------- | ------------------- |
| Algorithm           | Specialized solver (`lbfgs`, `liblinear`, etc.) | SGD                 |
| Small datasets      | Often better                                    | Fine                |
| Huge datasets       | Can be slower                                   | Usually faster      |
| Online learning     | No                                              | Yes (`partial_fit`) |
| Text classification | Good                                            | Very common         |

---

## A unique advantage: Online learning

`SGDClassifier` can learn incrementally.

```python
clf.partial_fit(X_batch, y_batch)
```

You can feed data in chunks instead of loading everything into memory.

This is useful when:

* Data arrives continuously
* Dataset is too large for RAM
* Streaming applications

---

### Connection to mlcourse.ai

In the IMDb sentiment analysis section, you may see `SGDClassifier` used because:

1. Text data has very high dimensionality.
2. The feature matrix is sparse.
3. SGD trains linear classifiers efficiently on such data.
4. It can approximate logistic regression (`loss='log_loss'`) or linear SVM (`loss='hinge'`) while scaling better to large datasets.

So, `SGDClassifier` is not a completely different type of model. It's mainly a **training strategy (SGD) for linear classifiers**, especially useful when datasets become large.


In [5]:
alphas = np.logspace(-4,0,20)
sgd_model = SGDClassifier(loss="hinge", n_jobs=-1, random_state=17)
logit_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2)),
    ("sgd_model", sgd_model)
])

val_train, val_test = validation_curve(
    estimator=logit_pipe, X=X, y=y, param_name="sgd_model__alpha", param_range=alphas, cv=5, scoring="roc_auc"
)
